In [0]:
# crear base de datos para gold
storage_path = dbutils.secrets.get(scope="tpfinal", key="keyfinal")  

gold_base = f"{storage_path}/gold"
gold_enriched_path = f"{gold_base}/vuelos_enriquecidos"
gold_metrics_path  = f"{gold_base}/metrics_daily"
spark.sql("CREATE DATABASE IF NOT EXISTS capa_gold")
spark.sql("USE capa_gold")

DataFrame[]

In [0]:
# carga de capa silver y filtrar rutas y periodo

from pyspark.sql import functions as F

v = spark.table("capa_silver.vuelos")
h = spark.table("capa_silver.feriados")

# Período del caso de estudio
start_date = "2024-12-01"
end_date   = "2025-04-30"

# Rutas AEP <-> (JUJ, SLA, TUC)
destinos = ["JUJ", "SLA", "TUC"]
filtro_rutas = (
    ((F.col("origin")=="AEP") & (F.col("destination").isin(destinos))) |
    ((F.col("destination")=="AEP") & (F.col("origin").isin(destinos)))
)

v_filtrado = (
    v.filter((F.col("flight_date") >= F.lit(start_date)) & (F.col("flight_date") <= F.lit(end_date)))
     .filter(filtro_rutas)
)

v_filtrado.select("flight_date","origin","destination","airline","flight_number").limit(5).display()


flight_date,origin,destination,airline,flight_number
2024-12-23,AEP,SLA,AEROLINEAS ARGENTINAS,AR 1488
2024-12-23,AEP,SLA,JETSMART AIRLINES,WJ 3012
2024-12-23,AEP,TUC,AEROLINEAS ARGENTINAS,AR 1468
2024-12-23,AEP,TUC,Flybondi,FO 5220
2024-12-23,AEP,JUJ,AEROLINEAS ARGENTINAS,AR 1514


In [0]:
# agregar feriados y derivados

# Join por fecha
vj = (v_filtrado.alias("v")
      .join(h.alias("h"), F.col("v.flight_date")==F.col("h.holiday_date"), "left"))

# Derivados
vj = (vj
    .withColumn("is_holiday", F.when(F.col("h.holiday_date").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("weekday_num", F.dayofweek("v.flight_date"))  # 1=Dom ... 7=Sáb (Databricks/Spark)
    .withColumn("weekday_name", F.date_format("v.flight_date", "E"))
    # Cap outliers de demora (opcional): si > 24h, lo ponemos null para no sesgar
    .withColumn("delay_minutes", F.when(F.col("v.delay_minutes") > 24*60, F.lit(None)).otherwise(F.col("v.delay_minutes")))
    # Cancelación: sin act_ts o campos estado que contengan 'CANC'
    .withColumn("cancel_flag",
        F.when(F.col("v.act_ts").isNull(), 1)
         .when(F.upper(F.col("v.estes")).like("%CANC%"), 1)
         .when(F.upper(F.col("v.estin")).like("%CANC%"), 1)
         .when(F.upper(F.col("v.estbr")).like("%CANC%"), 1)
         .otherwise(0)
    )
    # Selección de columnas ordenadas
    .select(
        F.col("v.flight_date"),
        F.col("v.sched_ts"), F.col("v.est_ts"), F.col("v.act_ts"),
        F.col("v.origin"), F.col("v.destination"),
        F.col("v.airline"), F.col("v.airline_code"), F.col("v.flight_number"),
        F.col("v.mov"), F.col("v.estes"), F.col("v.estin"), F.col("v.estbr"),
        "delay_minutes", "cancel_flag", "is_holiday", "weekday_num", "weekday_name",
        F.col("h.name").alias("holiday_name"), F.col("h.type").alias("holiday_type")
    )
)

vj.limit(10).display()


flight_date,sched_ts,est_ts,act_ts,origin,destination,airline,airline_code,flight_number,mov,estes,estin,estbr,delay_minutes,cancel_flag,is_holiday,weekday_num,weekday_name,holiday_name,holiday_type
2024-12-23,2024-12-23T04:05:00Z,null,null,AEP,SLA,AEROLINEAS ARGENTINAS,AR,AR 1488,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T04:45:00Z,null,null,AEP,SLA,JETSMART AIRLINES,WJ,WJ 3012,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T05:15:00Z,null,null,AEP,TUC,AEROLINEAS ARGENTINAS,AR,AR 1468,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T06:30:00Z,null,null,AEP,TUC,Flybondi,FO,FO 5220,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T06:40:00Z,null,null,AEP,JUJ,AEROLINEAS ARGENTINAS,AR,AR 1514,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T08:55:00Z,null,null,AEP,SLA,AEROLINEAS ARGENTINAS,AR,AR 1494,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T09:05:00Z,null,null,AEP,JUJ,AEROLINEAS ARGENTINAS,AR,AR 1512,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T09:20:00Z,null,null,AEP,TUC,AEROLINEAS ARGENTINAS,AR,AR 1472,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T10:55:00Z,null,null,AEP,TUC,JETSMART AIRLINES,WJ,WJ 3230,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T12:45:00Z,null,null,AEP,SLA,AEROLINEAS ARGENTINAS,AR,AR 1496,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null


In [0]:
# guardar datos enriquecidos

# Escribir a Delta
(vj.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema","true")
   .save(gold_enriched_path))

# Registrar tabla
spark.sql(f"""
CREATE TABLE IF NOT EXISTS capa_gold.vuelos_enriquecidos
USING delta
LOCATION '{gold_enriched_path}'
""")

spark.table("capa_gold.vuelos_enriquecidos").limit(10).display()


flight_date,sched_ts,est_ts,act_ts,origin,destination,airline,airline_code,flight_number,mov,estes,estin,estbr,delay_minutes,cancel_flag,is_holiday,weekday_num,weekday_name,holiday_name,holiday_type
2024-12-23,2024-12-23T04:05:00Z,null,null,AEP,SLA,AEROLINEAS ARGENTINAS,AR,AR 1488,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T04:45:00Z,null,null,AEP,SLA,JETSMART AIRLINES,WJ,WJ 3012,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T05:15:00Z,null,null,AEP,TUC,AEROLINEAS ARGENTINAS,AR,AR 1468,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T06:30:00Z,null,null,AEP,TUC,Flybondi,FO,FO 5220,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T06:40:00Z,null,null,AEP,JUJ,AEROLINEAS ARGENTINAS,AR,AR 1514,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T08:55:00Z,null,null,AEP,SLA,AEROLINEAS ARGENTINAS,AR,AR 1494,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T09:05:00Z,null,null,AEP,JUJ,AEROLINEAS ARGENTINAS,AR,AR 1512,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T09:20:00Z,null,null,AEP,TUC,AEROLINEAS ARGENTINAS,AR,AR 1472,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T10:55:00Z,null,null,AEP,TUC,JETSMART AIRLINES,WJ,WJ 3230,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null
2024-12-23,2024-12-23T12:45:00Z,null,null,AEP,SLA,AEROLINEAS ARGENTINAS,AR,AR 1496,D,En Horario,On Time,No Horario,null,1,0,2,Mon,null,null


In [0]:
from pyspark.sql import Window

g = spark.table("capa_gold.vuelos_enriquecidos")

# Agregaciones por día, ruta y aerolínea
metrics = (g.groupBy(
                "flight_date","origin","destination",
                "airline","is_holiday","weekday_num","weekday_name"
          )
          .agg(
              F.count("*").alias("flights"),
              F.avg("delay_minutes").alias("delay_mean"),
              F.sum("cancel_flag").alias("cancellations")
          )
          .withColumn("cancel_rate", F.col("cancellations")/F.col("flights"))
          .orderBy("flight_date","airline","origin","destination")
)

metrics.limit(10).display()


flight_date,origin,destination,airline,is_holiday,weekday_num,weekday_name,flights,delay_mean,cancellations,cancel_rate
2024-12-23,AEP,JUJ,AEROLINEAS ARGENTINAS,0,2,Mon,129,13.081632653061224,80,0.6201550387596899
2024-12-23,AEP,SLA,AEROLINEAS ARGENTINAS,0,2,Mon,301,9.46078431372549,199,0.6611295681063123
2024-12-23,AEP,TUC,AEROLINEAS ARGENTINAS,0,2,Mon,173,2.689655172413793,115,0.6647398843930635
2024-12-23,JUJ,AEP,AEROLINEAS ARGENTINAS,0,2,Mon,80,11.0,66,0.825
2024-12-23,SLA,AEP,AEROLINEAS ARGENTINAS,0,2,Mon,241,-6.5636363636363635,186,0.7717842323651453
2024-12-23,TUC,AEP,AEROLINEAS ARGENTINAS,0,2,Mon,200,-1.0961538461538463,148,0.74
2024-12-23,AEP,JUJ,Flybondi,0,2,Mon,43,71.0,41,0.9534883720930233
2024-12-23,AEP,SLA,Flybondi,0,2,Mon,40,66.0,39,0.975
2024-12-23,AEP,TUC,Flybondi,0,2,Mon,86,23.576923076923077,60,0.6976744186046512
2024-12-23,JUJ,AEP,Flybondi,0,2,Mon,67,410.92857142857144,53,0.7910447761194029


In [0]:
(metrics.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema","true")
   .save(gold_metrics_path))

spark.sql(f"""
CREATE TABLE IF NOT EXISTS capa_gold.metrics_daily
USING delta
LOCATION '{gold_metrics_path}'
""")

spark.table("capa_gold.metrics_daily").orderBy("flight_date").limit(20).display()

flight_date,origin,destination,airline,is_holiday,weekday_num,weekday_name,flights,delay_mean,cancellations,cancel_rate
2024-12-23,AEP,JUJ,AEROLINEAS ARGENTINAS,0,2,Mon,129,13.081632653061224,80,0.6201550387596899
2024-12-23,AEP,SLA,AEROLINEAS ARGENTINAS,0,2,Mon,301,9.46078431372549,199,0.6611295681063123
2024-12-23,AEP,TUC,AEROLINEAS ARGENTINAS,0,2,Mon,173,2.689655172413793,115,0.6647398843930635
2024-12-23,JUJ,AEP,AEROLINEAS ARGENTINAS,0,2,Mon,80,11.0,66,0.825
2024-12-23,SLA,AEP,AEROLINEAS ARGENTINAS,0,2,Mon,241,-6.5636363636363635,186,0.7717842323651453
2024-12-23,TUC,AEP,AEROLINEAS ARGENTINAS,0,2,Mon,200,-1.0961538461538463,148,0.74
2024-12-23,AEP,JUJ,Flybondi,0,2,Mon,43,71.0,41,0.9534883720930233
2024-12-23,AEP,SLA,Flybondi,0,2,Mon,40,66.0,39,0.975
2024-12-23,AEP,TUC,Flybondi,0,2,Mon,86,23.576923076923077,60,0.6976744186046512
2024-12-23,JUJ,AEP,Flybondi,0,2,Mon,67,410.92857142857144,53,0.7910447761194029


In [0]:
spark.sql("""
CREATE OR REPLACE VIEW capa_gold.vw_delay_cancel_by_airline AS
SELECT
  airline,
  is_holiday,
  ROUND(AVG(delay_mean), 2)  AS avg_delay_minutes,
  ROUND(AVG(cancel_rate), 4) AS avg_cancel_rate
FROM capa_gold.metrics_daily
GROUP BY airline, is_holiday
""")

DataFrame[]

In [0]:
%sql
select * from capa_gold.vw_delay_cancel_by_airline

airline,is_holiday,avg_delay_minutes,avg_cancel_rate
Flybondi,0,92.27,0.4328
Flybondi,1,60.8,0.4499
JETSMART AIRLINES,1,10.06,0.3129
JETSMART AIRLINES,0,4.13,0.3475
AEROLINEAS ARGENTINAS,0,3.91,0.3897
AEROLINEAS ARGENTINAS,1,-1.62,0.6666
